# PROYECTO DE AULA: MINERÍA DE DATOS - SEGUNDO CORTE (20%)
## Modelo Predictivo para la Detección Temprana de Diabetes Mellitus: EDA Profundo, Modelado y Validación Comparativa

### 1.1 Continuidad del Primer Avance
- **Título del proyecto:** Modelo predictivo para la detección temprana de diabetes mellitus mediante técnicas de minería de datos en variables clínicas y antropométricas.
- **Autor / Integrante:** Jorge Mario Camargo Pérez
- **Programa Académico:** Ingeniería de Sistemas - 6to Semestre
- **Institución:** Universidad de la Costa (CUC)
- **Asignatura:** Minería de Datos
- **Tipo de proyecto:** Proyecto de Clasificación (Aprendizaje Automático Supervisado).
- **Pregunta de investigación:** ¿Es posible clasificar de forma temprana y confiable si una paciente es diabética o no, minimizando los falsos negativos a partir de biomarcadores rutinarios no invasivos?
- **Dataset:** *Pima Indians Diabetes Database* (National Institute of Diabetes and Digestive and Kidney Diseases - NIDDK).
- **Fuente y descarga:** Repositorio Kaggle / UCI Machine Learning Repository (`diabetes.csv`).
- **Variable objetivo:** `Outcome` (Binaria: `0` = Sin diabetes / Negativo, `1` = Con diabetes / Positivo).
- **Justificación de continuidad:** Se conserva el problema y el dataset del Corte 1, profundizando en la preparación robusta de datos, análisis bivariado/multivariado de correlación y riesgo, entrenamiento de modelos supervisados (Regresión Logística balanceada vs. Árbol de Decisión regularizado), evaluación exhaustiva de matrices de confusión y validación cruzada estratificada.

## 1. Preparación del Entorno e Importación de Librerías

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report, roc_curve, auc
)

# Configuración gráfica institucional
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 10
%matplotlib inline

SEMILLA = 42

## 2. Carga y Diagnóstico Inicial de Datos (Pipeline de Limpieza)
Retomamos el dataset original de 768 instancias y 9 atributos. Verificamos la calidad y resolvemos la presencia de ceros fisiológicos no viables en variables clínicas.

In [ ]:
# Carga de datos (con fallback local)
url_github = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv'
columnas = [
    'Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
    'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'
]

try:
    df_raw = pd.read_csv(url_github, names=columnas)
    print('Dataset cargado exitosamente desde repositorio web.')
except Exception:
    df_raw = pd.read_csv('diabetes.csv')
    print('Dataset cargado desde archivo local.')

print(f'Dimensiones iniciales: {df_raw.shape[0]} registros y {df_raw.shape[1]} columnas.')
print(f'Registros duplicados: {df_raw.duplicated().sum()}')
df_raw.head()

### Tratamiento de Ceros Fisiológicos e Imputación Estadística
En atributos biomédicos como `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin` y `BMI`, valores iguales a 0 representan datos ausentes (*missing values* encubiertos). Se reemplazan por `NaN` y se imputan con la mediana para evitar sesgos por asimetría y preservar el tamaño muestral sin recurrir a la eliminación ingenua de filas.

In [ ]:
cols_biologicas = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
ceros_detectados = (df_raw[cols_biologicas] == 0).sum()
print('Conteo de ceros fisiológicos encubiertos por variable:\n', ceros_detectados)

df_clean = df_raw.copy()
for col in cols_biologicas:
    df_clean[col] = df_clean[col].replace(0, np.nan)
    mediana_val = df_clean[col].median()
    df_clean[col] = df_clean[col].fillna(mediana_val)

print(f'\nTotal de nulos pendientes tras imputación estadística: {df_clean.isnull().sum().sum()}')
df_clean.describe().round(2)

## 3. Análisis Exploratorio de Datos (EDA Profundo)
Exploramos la distribución de la variable objetivo, contrastes por diagnóstico, dispersiones y la matriz de correlación de Pearson para guiar las decisiones de modelado.

In [ ]:
# 3.1 Distribución de la variable objetivo y balance de clases
conteo = df_clean['Outcome'].value_counts()
porcentajes = df_clean['Outcome'].value_counts(normalize=True) * 100

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(x=conteo.index, y=conteo.values, palette=['#4C78A8', '#E15759'], ax=ax)
ax.set_title('Figura 1. Distribución de Clases (0: No Diabético vs 1: Diabético)', fontweight='bold')
ax.set_xticklabels(['Negativo (0)', 'Positivo (1)'])
ax.set_ylabel('Número de Pacientes')
for i, p in enumerate(ax.patches):
    ax.annotate(f'{conteo.iloc[i]} ({porcentajes.iloc[i]:.1f}%)',
                (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                ha='center', va='center', color='white', fontweight='bold')
plt.tight_layout()
plt.show()

> **Interpretación:** La clase 0 representa el 65.1% (500 pacientes) y la clase 1 el 34.9% (268 pacientes). Este desbalance moderado exige el uso de hiperparámetros de compensación (`class_weight='balanced'`) y métricas más informativas que el simple accuracy (como Recall y F1-score).

In [ ]:
# 3.2 Análisis Bivariado: Biomarcadores críticos vs Condición Diagnóstica
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

sns.boxplot(data=df_clean, x='Outcome', y='Glucose', palette=['#4C78A8', '#E15759'], ax=axes[0])
axes[0].set_title('Glucosa plasmática (mg/dL)', fontweight='bold')
axes[0].set_xticklabels(['No Diabético', 'Diabético'])

sns.boxplot(data=df_clean, x='Outcome', y='BMI', palette=['#4C78A8', '#E15759'], ax=axes[1])
axes[1].set_title('Índice de Masa Corporal (kg/m²)', fontweight='bold')
axes[1].set_xticklabels(['No Diabético', 'Diabético'])

sns.boxplot(data=df_clean, x='Outcome', y='Age', palette=['#4C78A8', '#E15759'], ax=axes[2])
axes[2].set_title('Edad cronológica (Años)', fontweight='bold')
axes[2].set_xticklabels(['No Diabético', 'Diabético'])

plt.suptitle('Figura 2. Comparación de Indicadores Principales según Diagnóstico', fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

> **Interpretación:** Las pacientes diabéticas exhiben medianas significativamente mayores en glucosa plasmática (~140 mg/dL frente a ~108 mg/dL), mayor adiposidad (IMC superior a 34 kg/m²) y edad promedio más avanzada. Estas tres variables perfilan una clara capacidad de discriminación.

In [ ]:
# 3.3 Matriz de Correlación Lineal de Pearson
plt.figure(figsize=(9, 7))
corr = df_clean.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='vlag', center=0, linewidths=0.5)
plt.title('Figura 3. Matriz de Correlación de Atributos Clínicos', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

> **Interpretación:** La glucosa tiene la correlación lineal más fuerte con el diagnóstico (`Outcome`, r ≈ 0.49), seguida por el IMC (r ≈ 0.31) y la edad (r ≈ 0.24). No se observan correlaciones inter-variables excesivas (>0.80) que indiquen problemas de colinealidad destructiva.

## 4. Selección, Entrenamiento y Validación de Modelos
### 4.1 Partición Estratificada y Escalamiento
Separamos las variables predictoras ($X$) de la variable dependiente ($y$). Aplicamos una partición del 75% para entrenamiento y 25% para prueba manteniendo la proporción de clases mediante estratificación (`stratify=y`).

In [ ]:
X = df_clean.drop(columns=['Outcome'])
y = df_clean['Outcome']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEMILLA, stratify=y
)

print(f'Muestra de Entrenamiento: {X_train.shape[0]} pacientes')
print(f'Muestra de Prueba: {X_test.shape[0]} pacientes')
print(f'Distribución en prueba (Clase 0 / 1): {np.bincount(y_test)}')

### 4.2 Entrenamiento de Modelos Representativos
Se implementan dos enfoques con filosofías algorítmicas distintas:
1. **Regresión Logística Balanceada:** Modelo probabilístico lineal con penalización $L_2$ y ajuste de pesos de clase para compensar la menor prevalencia de positivos.
2. **Árbol de Decisión Regularizado:** Clasificador no paramétrico con reglas ortogonales, acotado a profundidad máxima de 4 y tamaño mínimo de hoja de 10 muestras para prevenir sobreajuste.

In [ ]:
# 1. Pipeline Regresión Logística
pipe_logistica = Pipeline([
    ('escalador', StandardScaler()),
    ('clasificador', LogisticRegression(class_weight='balanced', random_state=SEMILLA, max_iter=1000))
])
pipe_logistica.fit(X_train, y_train)
pred_logistica = pipe_logistica.predict(X_test)

# 2. Pipeline Árbol de Decisión
pipe_arbol = Pipeline([
    ('escalador', StandardScaler()),
    ('clasificador', DecisionTreeClassifier(max_depth=4, min_samples_leaf=10, class_weight='balanced', random_state=SEMILLA))
])
pipe_arbol.fit(X_train, y_train)
pred_arbol = pipe_arbol.predict(X_test)

print('Modelos entrenados y predicciones generadas.')

## 5. Evaluación e Interpretación Comparativa
Calculamos Accuracy, Precision, Recall y F1-score en el conjunto de prueba independiente.

In [ ]:
def calcular_metricas(y_real, y_pred):
    return {
        'Accuracy': round(accuracy_score(y_real, y_pred), 3),
        'Precision': round(precision_score(y_real, y_pred), 3),
        'Recall (Sensibilidad)': round(recall_score(y_real, y_pred), 3),
        'F1-score': round(f1_score(y_real, y_pred), 3)
    }

tabla_metricas = pd.DataFrame([
    {'Modelo': 'Regresión Logística', **calcular_metricas(y_test, pred_logistica)},
    {'Modelo': 'Árbol de Decisión', **calcular_metricas(y_test, pred_arbol)}
]).set_index('Modelo')

print('Tabla 1. Comparación de Métricas en Conjunto de Prueba')
tabla_metricas

In [ ]:
# 5.1 Matrices de Confusión
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
orden = ['Negativo (0)', 'Positivo (1)']

cm_lr = confusion_matrix(y_test, pred_logistica)
disp_lr = ConfusionMatrixDisplay(confusion_matrix=cm_lr, display_labels=orden)
disp_lr.plot(ax=axes[0], cmap='Blues', values_format='d')
axes[0].set_title('Matriz: Regresión Logística', fontweight='bold')
axes[0].grid(False)

cm_dt = confusion_matrix(y_test, pred_arbol)
disp_dt = ConfusionMatrixDisplay(confusion_matrix=cm_dt, display_labels=orden)
disp_dt.plot(ax=axes[1], cmap='Oranges', values_format='d')
axes[1].set_title('Matriz: Árbol de Decisión', fontweight='bold')
axes[1].grid(False)

plt.suptitle('Figura 4. Matrices de Confusión en Prueba (N=192)', fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

> **Interpretación de errores clínicos:** En salud, el error más costoso es el **Falso Negativo** (paciente enferma diagnosticada como sana, quien pierde la oportunidad de tratamiento oportuno). La Regresión Logística arrojó 16 falsos negativos frente a 21 del Árbol de Decisión, logrando una mayor sensibilidad clínica (Recall de 76.1% vs 68.7%).

In [ ]:
# 5.2 Validación Cruzada Estratificada (5 Folds)
cv_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
cv_scores_lr = cross_val_score(pipe_logistica, X, y, cv=cv_kfold, scoring='f1')
cv_scores_dt = cross_val_score(pipe_arbol, X, y, cv=cv_kfold, scoring='f1')

resumen_cv = pd.DataFrame({
    'Modelo': ['Regresión Logística', 'Árbol de Decisión'],
    'F1 Promedio (CV=5)': [round(cv_scores_lr.mean(), 3), round(cv_scores_dt.mean(), 3)],
    'Desviación Estándar': [round(cv_scores_lr.std(), 3), round(cv_scores_dt.std(), 3)]
}).set_index('Modelo')

print('Tabla 2. Validación Cruzada Estratificada (F1-score)')
resumen_cv

In [ ]:
# 5.3 Importancia de Variables en el Árbol de Decisión
arbol_modelo = pipe_arbol.named_steps['clasificador']
importancias = pd.Series(arbol_modelo.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8, 4.5))
sns.barplot(x=importancias.values, y=importancias.index, palette='crest')
plt.title('Figura 5. Importancia Relativa de Variables (Gini Reduction)', fontweight='bold')
plt.xlabel('Importancia')
plt.ylabel('Variable Clínica')
plt.tight_layout()
plt.show()
importancias.round(3)

## 6. Conclusiones del Notebook
1. **Hallazgos del EDA:** La glucosa sérica (`Glucose`), el índice de masa corporal (`BMI`) y la `Age` constituyen los tres biomarcadores más informativos, evidenciando correlaciones positivas con la presencia de diabetes y diferencias estadísticamente marcadas en sus medianas.
2. **Modelo seleccionado:** Se selecciona la **Regresión Logística balanceada** como la arquitectura más idónea para esta etapa.
3. **Evidencias técnicas:** Superó al árbol en el conjunto de prueba tanto en exactitud global (Accuracy: 0.786 vs 0.781) como en F1-score (0.713 vs 0.687), y demostró mayor estabilidad en validación cruzada (F1: 0.760 ± 0.044 vs 0.730 ± 0.024).
4. **Criterio clínico y limitaciones:** La regresión logística redujo a 16 los falsos negativos (Recall de 76.1%), frente a 21 del árbol de decisión. La principal limitación identificada radica en la colinealidad parcial entre la edad y los embarazos, así como el 24% de falsos positivos que requerirán técnicas de ensamble (Random Forest) y optimización de umbrales para el Corte 3.